# Bulk Download Water Station Time Series

## 📖 Summary

This notebook downloads **complete historical time series data (2016-2024)** for all reliable water monitoring stations identified in the parameter discovery phase. The download uses year-by-year chunked queries to avoid API rate limiting and includes mass balance calculations for Waal and IJssel discharge.

**Input:** `water_stations_for_modeling.gpkg` (66 water height + 14 discharge stations)

**Output:** Clean Parquet files ready for feature engineering

## 🎯 Objective

Download and prepare raw time series data from Rijkswaterstaat's API for all reliable stations, implementing necessary calculations and quality filtering.

## 📋 Process Flow

1. **Load station lists** from GeoPackage
2. **Download water height** - 66 stations × 9 years (chunked)
3. **Download discharge** - Base stations (Lobith, Pannerden, Driel, etc.)
4. **Calculate Waal discharge** - `Q_waal = Q_lobith - Q_pannerden`
5. **Calculate IJssel discharge** - `Q_ijssel = Q_pannerden - Q_driel`
6. **Data quality filtering** - Remove error codes (`999999999`)
7. **Save to Parquet** - Efficient compressed format
8. **Validation summary** - Statistics and completeness check

## 🎁 Deliverables

- **`timeseries/water_height/{station_code}.parquet`** - 66 files
- **`timeseries/discharge/{station_code}.parquet`** - 14 files (including calculated)
- **`download_log.csv`** - Success/failure tracking
- **`data_quality_report.csv`** - Completeness and error statistics

## ⏱️ Estimated Runtime

- Water height: ~45-60 minutes (66 stations × 9 years × 0.3s/call)
- Discharge: ~15-20 minutes (14 stations × 9 years × 0.3s/call)
- **Total: ~60-80 minutes** (with API throttling)

---

## 1. Setup & Configuration

In [1]:
# Imports
import sys
sys.path.insert(0, "..")
sys.path.append("../..")

import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
import requests
from datetime import datetime
import time
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports successful!")

✅ Imports successful!


/Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Import paths
import src.paths as PATHS

In [3]:
# Configuration
INPUT_GPKG = PATHS.DATA_DIR / "water_stations_by_parameter" / "water_stations_for_modeling.gpkg"
OUTPUT_DIR = PATHS.DATA_DIR / "water_stations_timeseries"
OUTPUT_DIR.mkdir(exist_ok=True)

# Create subdirectories
WATER_HEIGHT_DIR = OUTPUT_DIR / "water_height"
DISCHARGE_DIR = OUTPUT_DIR / "discharge"
WATER_HEIGHT_DIR.mkdir(exist_ok=True)
DISCHARGE_DIR.mkdir(exist_ok=True)

# API Configuration
BASE_URL = "https://ddapi20-waterwebservices.rijkswaterstaat.nl"
OBSERVATIONS_ENDPOINT = f"{BASE_URL}/ONLINEWAARNEMINGENSERVICES/OphalenWaarnemingen"
HEADERS = {
    "Content-Type": "application/json",
    "X-API-KEY": "dummy-key"
}

# Download parameters
START_YEAR = 2016
END_YEAR = 2024
YEARS = list(range(START_YEAR, END_YEAR + 1))
API_DELAY = 0.3  # seconds between API calls
REQUEST_TIMEOUT = 30  # seconds

# Data quality
ERROR_CODE = 999999999.0  # Standard RWS error code

print(f"📂 Input GeoPackage: {INPUT_GPKG}")
print(f"📁 Output directory: {OUTPUT_DIR}")
print(f"📅 Download period: {START_YEAR}-{END_YEAR} ({len(YEARS)} years)")
print(f"⏱️  API delay: {API_DELAY}s between calls")
print(f"🔧 Error code to filter: {ERROR_CODE}")

📂 Input GeoPackage: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_by_parameter/water_stations_for_modeling.gpkg
📁 Output directory: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries
📅 Download period: 2016-2024 (9 years)
⏱️  API delay: 0.3s between calls
🔧 Error code to filter: 999999999.0


## 2. Load Station Lists from GeoPackage

In [4]:
print("📥 Loading station lists from GeoPackage...\n")

# Load water height stations
water_height_stations = gpd.read_file(INPUT_GPKG, layer='water_height_stations')
wathte_codes = water_height_stations['CODE'].unique().tolist()

print(f"🌊 Water height stations: {len(wathte_codes)}")
print(f"   Sample: {wathte_codes[:5]}\n")

# Load discharge stations
discharge_stations = gpd.read_file(INPUT_GPKG, layer='discharge_stations')
discharge_codes = discharge_stations['CODE'].unique().tolist()

print(f"💧 Discharge stations: {len(discharge_codes)}")
print(f"   Sample: {discharge_codes[:5]}\n")

# Load mass balance metadata
try:
    calc_metadata = gpd.read_file(INPUT_GPKG, layer='discharge_calculation_metadata')
    print(f"📋 Mass balance metadata loaded:")
    for _, row in calc_metadata.iterrows():
        print(f"   • {row['target_station']}: {row['calculation_method']}")
        print(f"     Valid period: {row['valid_period']}")
    print()
except Exception as e:
    print(f"⚠️  No mass balance metadata found: {e}\n")
    calc_metadata = None

print(f"✅ Station lists loaded successfully!")

📥 Loading station lists from GeoPackage...

🌊 Water height stations: 66
   Sample: ['negenoord.oost', 'elsloo.maas', 'nijmegen.waal', 'negenoord.west', 'kampen.ijssel']

💧 Discharge stations: 13
   Sample: ['maastricht.borgharen.maas.beneden', 'genemuiden', 'megen.maas', 'pannerden.pannerdenschkanaal', 'lobith.bovenrijn.tolkamer']

📋 Mass balance metadata loaded:
   • millingenaanderijn: Q_waal = Q_lobith - Q_pannerden
     Valid period: 2016-2024

✅ Station lists loaded successfully!


## 3. Define Download Functions

In [5]:
def fetch_year_data(station_code, parameter_code, year, delay=0.3):
    """
    Fetch time series data for a single station, parameter, and year.
    
    Args:
        station_code: Station identifier (e.g., 'lobith.bovenrijn.tolkamer')
        parameter_code: Parameter code ('WATHTE' or 'Q')
        year: Year to download (e.g., 2020)
        delay: Seconds to wait after request (default 0.3)
    
    Returns:
        DataFrame with columns: timestamp, value, station_code, year
        Or None if no data/error
    """
    body = {
        "Locatie": {"Code": station_code},
        "AquoPlusWaarnemingMetadata": {
            "AquoMetadata": {
                "Compartiment": {"Code": "OW"},
                "Grootheid": {"Code": parameter_code},
                "ProcesType": "meting"
            }
        },
        "Periode": {
            "Begindatumtijd": f"{year}-01-01T00:00:00.000+01:00",
            "Einddatumtijd": f"{year}-12-31T23:59:59.999+01:00"
        }
    }
    
    try:
        response = requests.post(
            OBSERVATIONS_ENDPOINT, 
            json=body, 
            headers=HEADERS, 
            timeout=REQUEST_TIMEOUT
        )
        
        if response.status_code == 200:
            data = response.json()
            if data.get('Succesvol') and data.get('WaarnemingenLijst'):
                metingen = data['WaarnemingenLijst'][0].get('MetingenLijst', [])
                if metingen:
                    records = []
                    for m in metingen:
                        records.append({
                            'timestamp': m['Tijdstip'],
                            'value': m['Meetwaarde'].get('Waarde_Numeriek'),
                            'station_code': station_code,
                            'year': year
                        })
                    
                    df = pd.DataFrame(records)
                    df['timestamp'] = pd.to_datetime(df['timestamp'])
                    
                    time.sleep(delay)
                    return df
        
        time.sleep(delay)
        return None
        
    except Exception as e:
        time.sleep(delay)
        return None


def download_station_timeseries(station_code, parameter_code, years, desc=""):
    """
    Download complete time series for a station across multiple years.
    
    Args:
        station_code: Station identifier
        parameter_code: 'WATHTE' or 'Q'
        years: List of years to download
        desc: Description for progress bar
    
    Returns:
        DataFrame with all years concatenated, or None if all years failed
    """
    all_data = []
    
    for year in tqdm(years, desc=desc, leave=False):
        year_df = fetch_year_data(station_code, parameter_code, year)
        if year_df is not None and len(year_df) > 0:
            all_data.append(year_df)
    
    if all_data:
        combined = pd.concat(all_data, ignore_index=True)
        combined = combined.sort_values('timestamp').reset_index(drop=True)
        return combined
    else:
        return None


def filter_error_codes(df, value_column='value', error_code=999999999.0):
    """
    Remove error codes from time series data.
    
    Args:
        df: DataFrame with time series
        value_column: Name of column containing values
        error_code: Error code to remove
    
    Returns:
        Filtered DataFrame and count of removed values
    """
    if df is None or len(df) == 0:
        return df, 0
    
    original_count = len(df)
    df_clean = df[df[value_column] != error_code].copy()
    removed_count = original_count - len(df_clean)
    
    return df_clean, removed_count


print("✅ Download functions defined!")

✅ Download functions defined!


## 4. Download Water Height Time Series

Download all 66 water height stations (2016-2024).

In [6]:
print("="*80)
print("DOWNLOADING WATER HEIGHT TIME SERIES")
print("="*80)

print(f"\n📊 Stations to download: {len(wathte_codes)}")
print(f"📅 Years: {START_YEAR}-{END_YEAR}")
print(f"🔢 Total API calls: {len(wathte_codes)} × {len(YEARS)} = {len(wathte_codes) * len(YEARS):,}")
print(f"⏱️  Estimated time: ~{(len(wathte_codes) * len(YEARS) * API_DELAY / 60):.0f} minutes\n")

print("🚀 Starting download...\n")

wathte_log = []
wathte_successful = 0
wathte_failed = 0

for station_code in tqdm(wathte_codes, desc="Water Height Stations"):
    # Download
    df = download_station_timeseries(
        station_code, 
        'WATHTE', 
        YEARS,
        desc=f"{station_code[:30]}"
    )
    
    if df is not None and len(df) > 0:
        # Filter error codes
        df_clean, removed_count = filter_error_codes(df, 'value', ERROR_CODE)
        
        if len(df_clean) > 0:
            # Rename column for clarity
            df_clean = df_clean.rename(columns={'value': 'water_level_cm'})
            
            # Save to Parquet
            output_path = WATER_HEIGHT_DIR / f"{station_code}.parquet"
            df_clean.to_parquet(output_path, index=False, compression='snappy')
            
            wathte_successful += 1
            wathte_log.append({
                'station_code': station_code,
                'parameter': 'WATHTE',
                'status': 'success',
                'total_measurements': len(df_clean),
                'removed_errors': removed_count,
                'years_covered': df_clean['year'].nunique(),
                'date_range': f"{df_clean['timestamp'].min()} to {df_clean['timestamp'].max()}",
                'file_path': str(output_path)
            })
        else:
            wathte_failed += 1
            wathte_log.append({
                'station_code': station_code,
                'parameter': 'WATHTE',
                'status': 'failed_all_errors',
                'total_measurements': 0,
                'removed_errors': removed_count,
                'years_covered': 0,
                'date_range': 'N/A',
                'file_path': 'N/A'
            })
    else:
        wathte_failed += 1
        wathte_log.append({
            'station_code': station_code,
            'parameter': 'WATHTE',
            'status': 'failed_no_data',
            'total_measurements': 0,
            'removed_errors': 0,
            'years_covered': 0,
            'date_range': 'N/A',
            'file_path': 'N/A'
        })

print("\n" + "="*80)
print("WATER HEIGHT DOWNLOAD COMPLETE")
print("="*80)
print(f"\n✅ Successful: {wathte_successful}/{len(wathte_codes)}")
print(f"❌ Failed: {wathte_failed}/{len(wathte_codes)}")
print(f"📂 Saved to: {WATER_HEIGHT_DIR}\n")

DOWNLOADING WATER HEIGHT TIME SERIES

📊 Stations to download: 66
📅 Years: 2016-2024
🔢 Total API calls: 66 × 9 = 594
⏱️  Estimated time: ~3 minutes

🚀 Starting download...



Water Height Stations: 100%|██████████| 66/66 [59:59<00:00, 54.54s/it]   


WATER HEIGHT DOWNLOAD COMPLETE

✅ Successful: 66/66
❌ Failed: 0/66
📂 Saved to: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/water_height



## 5. Download Base Discharge Stations

Download discharge data for all stations EXCEPT Millingen and Westervoort (which will be calculated).

In [7]:
print("="*80)
print("DOWNLOADING BASE DISCHARGE STATIONS")
print("="*80)

# Identify stations to download directly vs calculate
calculated_stations = ['millingenaanderijn', 'westervoort']
base_discharge_codes = [code for code in discharge_codes if code not in calculated_stations]

print(f"\n📊 Base discharge stations: {len(base_discharge_codes)}")
print(f"🧮 Calculated stations: {calculated_stations}")
print(f"📅 Years: {START_YEAR}-{END_YEAR}")
print(f"🔢 Total API calls: {len(base_discharge_codes)} × {len(YEARS)} = {len(base_discharge_codes) * len(YEARS):,}")
print(f"⏱️  Estimated time: ~{(len(base_discharge_codes) * len(YEARS) * API_DELAY / 60):.0f} minutes\n")

print("🚀 Starting download...\n")

discharge_log = []
discharge_successful = 0
discharge_failed = 0
discharge_data = {}  # Store in memory for mass balance calculations

for station_code in tqdm(base_discharge_codes, desc="Base Discharge Stations"):
    # Download
    df = download_station_timeseries(
        station_code, 
        'Q', 
        YEARS,
        desc=f"{station_code[:30]}"
    )
    
    if df is not None and len(df) > 0:
        # Filter error codes
        df_clean, removed_count = filter_error_codes(df, 'value', ERROR_CODE)
        
        if len(df_clean) > 0:
            # Rename column for clarity
            df_clean = df_clean.rename(columns={'value': 'discharge_m3s'})
            
            # Store for potential mass balance calculations
            discharge_data[station_code] = df_clean.copy()
            
            # Save to Parquet
            output_path = DISCHARGE_DIR / f"{station_code}.parquet"
            df_clean.to_parquet(output_path, index=False, compression='snappy')
            
            discharge_successful += 1
            discharge_log.append({
                'station_code': station_code,
                'parameter': 'Q',
                'status': 'success',
                'total_measurements': len(df_clean),
                'removed_errors': removed_count,
                'years_covered': df_clean['year'].nunique(),
                'date_range': f"{df_clean['timestamp'].min()} to {df_clean['timestamp'].max()}",
                'file_path': str(output_path)
            })
        else:
            discharge_failed += 1
            discharge_log.append({
                'station_code': station_code,
                'parameter': 'Q',
                'status': 'failed_all_errors',
                'total_measurements': 0,
                'removed_errors': removed_count,
                'years_covered': 0,
                'date_range': 'N/A',
                'file_path': 'N/A'
            })
    else:
        discharge_failed += 1
        discharge_log.append({
            'station_code': station_code,
            'parameter': 'Q',
            'status': 'failed_no_data',
            'total_measurements': 0,
            'removed_errors': 0,
            'years_covered': 0,
            'date_range': 'N/A',
            'file_path': 'N/A'
        })

print("\n" + "="*80)
print("BASE DISCHARGE DOWNLOAD COMPLETE")
print("="*80)
print(f"\n✅ Successful: {discharge_successful}/{len(base_discharge_codes)}")
print(f"❌ Failed: {discharge_failed}/{len(base_discharge_codes)}")
print(f"📂 Saved to: {DISCHARGE_DIR}\n")

DOWNLOADING BASE DISCHARGE STATIONS

📊 Base discharge stations: 12
🧮 Calculated stations: ['millingenaanderijn', 'westervoort']
📅 Years: 2016-2024
🔢 Total API calls: 12 × 9 = 108
⏱️  Estimated time: ~1 minutes

🚀 Starting download...



Base Discharge Stations: 100%|██████████| 12/12 [11:46<00:00, 58.89s/it]


BASE DISCHARGE DOWNLOAD COMPLETE

✅ Successful: 12/12
❌ Failed: 0/12
📂 Saved to: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/discharge



## 6. Calculate Waal Discharge (Mass Balance)

Calculate `Q_waal = Q_lobith - Q_pannerden` and attribute to Millingen a/d Rijn.

In [8]:
print("="*80)
print("CALCULATING WAAL DISCHARGE (MASS BALANCE)")
print("="*80)

print("\n🔬 Formula: Q_waal = Q_lobith - Q_pannerden\n")

# Required stations
lobith_code = 'lobith.bovenrijn.tolkamer'
pannerden_code = 'pannerden.pannerdenschkanaal'

if lobith_code in discharge_data and pannerden_code in discharge_data:
    print(f"✅ Found required stations:")
    print(f"   • {lobith_code}: {len(discharge_data[lobith_code]):,} measurements")
    print(f"   • {pannerden_code}: {len(discharge_data[pannerden_code]):,} measurements\n")
    
    # Merge on timestamp
    lobith_df = discharge_data[lobith_code][['timestamp', 'discharge_m3s']].copy()
    pannerden_df = discharge_data[pannerden_code][['timestamp', 'discharge_m3s']].copy()
    
    merged = pd.merge(
        lobith_df,
        pannerden_df,
        on='timestamp',
        how='inner',
        suffixes=('_lobith', '_pannerden')
    )
    
    print(f"🔗 Merged timestamps: {len(merged):,}\n")
    
    # Calculate Waal discharge
    merged['discharge_m3s'] = merged['discharge_m3s_lobith'] - merged['discharge_m3s_pannerden']
    
    # Add metadata
    merged['station_code'] = 'millingenaanderijn'
    merged['year'] = merged['timestamp'].dt.year
    merged['calculation_method'] = 'mass_balance'
    merged['source_lobith'] = lobith_code
    merged['source_pannerden'] = pannerden_code
    
    # Keep only needed columns
    waal_df = merged[['timestamp', 'discharge_m3s', 'station_code', 'year', 'calculation_method']].copy()
    
    # Validate
    negative_count = (waal_df['discharge_m3s'] < 0).sum()
    if negative_count > 0:
        print(f"⚠️  WARNING: {negative_count:,} negative values ({negative_count/len(waal_df)*100:.2f}%)")
        print(f"   Removing negative values...\n")
        waal_df = waal_df[waal_df['discharge_m3s'] >= 0].copy()
    
    print(f"📊 Waal discharge statistics:")
    print(f"   Measurements: {len(waal_df):,}")
    print(f"   Min:  {waal_df['discharge_m3s'].min():6.0f} m³/s")
    print(f"   Max:  {waal_df['discharge_m3s'].max():6.0f} m³/s")
    print(f"   Mean: {waal_df['discharge_m3s'].mean():6.0f} m³/s")
    print(f"   Std:  {waal_df['discharge_m3s'].std():6.0f} m³/s")
    print(f"   Years: {waal_df['year'].min()}-{waal_df['year'].max()}\n")
    
    # Save
    output_path = DISCHARGE_DIR / "millingenaanderijn.parquet"
    waal_df.to_parquet(output_path, index=False, compression='snappy')
    
    discharge_log.append({
        'station_code': 'millingenaanderijn',
        'parameter': 'Q',
        'status': 'calculated_mass_balance',
        'total_measurements': len(waal_df),
        'removed_errors': negative_count,
        'years_covered': waal_df['year'].nunique(),
        'date_range': f"{waal_df['timestamp'].min()} to {waal_df['timestamp'].max()}",
        'file_path': str(output_path)
    })
    
    print(f"✅ Waal discharge calculated and saved!")
    print(f"📂 {output_path}\n")
    
else:
    print(f"❌ Cannot calculate Waal discharge - missing required stations")
    print(f"   Required: {lobith_code}, {pannerden_code}")
    print(f"   Available: {list(discharge_data.keys())}\n")
    
    discharge_log.append({
        'station_code': 'millingenaanderijn',
        'parameter': 'Q',
        'status': 'failed_missing_sources',
        'total_measurements': 0,
        'removed_errors': 0,
        'years_covered': 0,
        'date_range': 'N/A',
        'file_path': 'N/A'
    })

CALCULATING WAAL DISCHARGE (MASS BALANCE)

🔬 Formula: Q_waal = Q_lobith - Q_pannerden

✅ Found required stations:
   • lobith.bovenrijn.tolkamer: 445,110 measurements
   • pannerden.pannerdenschkanaal: 3,288 measurements

🔗 Merged timestamps: 3,092

⚠️  WARNING: 6 negative values (0.19%)
   Removing negative values...

📊 Waal discharge statistics:
   Measurements: 3,086
   Min:     541 m³/s
   Max:  99999900000000004159462354073616908288 m³/s
   Mean: 648087491898898362526353339803762688 m³/s
   Std:  8025555339849759770754961992130232320 m³/s
   Years: 2016-2024

✅ Waal discharge calculated and saved!
📂 /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/discharge/millingenaanderijn.parquet



## 7. Calculate IJssel Discharge (Mass Balance)

Calculate `Q_ijssel = Q_pannerden - Q_driel` and attribute to Westervoort.

In [9]:
print("="*80)
print("CALCULATING IJSSEL DISCHARGE (MASS BALANCE)")
print("="*80)

print("\n🔬 Formula: Q_ijssel = Q_pannerden - Q_driel\n")

# Required stations
pannerden_code = 'pannerden.pannerdenschkanaal'
driel_code = 'driel.boven'

if pannerden_code in discharge_data and driel_code in discharge_data:
    print(f"✅ Found required stations:")
    print(f"   • {pannerden_code}: {len(discharge_data[pannerden_code]):,} measurements")
    print(f"   • {driel_code}: {len(discharge_data[driel_code]):,} measurements\n")
    
    # Merge on timestamp
    pannerden_df = discharge_data[pannerden_code][['timestamp', 'discharge_m3s']].copy()
    driel_df = discharge_data[driel_code][['timestamp', 'discharge_m3s']].copy()
    
    merged = pd.merge(
        pannerden_df,
        driel_df,
        on='timestamp',
        how='inner',
        suffixes=('_pannerden', '_driel')
    )
    
    print(f"🔗 Merged timestamps: {len(merged):,}\n")
    
    # Calculate IJssel discharge
    merged['discharge_m3s'] = merged['discharge_m3s_pannerden'] - merged['discharge_m3s_driel']
    
    # Add metadata
    merged['station_code'] = 'westervoort'
    merged['year'] = merged['timestamp'].dt.year
    merged['calculation_method'] = 'mass_balance'
    merged['source_pannerden'] = pannerden_code
    merged['source_driel'] = driel_code
    
    # Keep only needed columns
    ijssel_df = merged[['timestamp', 'discharge_m3s', 'station_code', 'year', 'calculation_method']].copy()
    
    # Validate
    negative_count = (ijssel_df['discharge_m3s'] < 0).sum()
    if negative_count > 0:
        print(f"⚠️  WARNING: {negative_count:,} negative values ({negative_count/len(ijssel_df)*100:.2f}%)")
        print(f"   Removing negative values...\n")
        ijssel_df = ijssel_df[ijssel_df['discharge_m3s'] >= 0].copy()
    
    print(f"📊 IJssel discharge statistics:")
    print(f"   Measurements: {len(ijssel_df):,}")
    print(f"   Min:  {ijssel_df['discharge_m3s'].min():6.0f} m³/s")
    print(f"   Max:  {ijssel_df['discharge_m3s'].max():6.0f} m³/s")
    print(f"   Mean: {ijssel_df['discharge_m3s'].mean():6.0f} m³/s")
    print(f"   Std:  {ijssel_df['discharge_m3s'].std():6.0f} m³/s")
    print(f"   Years: {ijssel_df['year'].min()}-{ijssel_df['year'].max()}\n")
    
    # Save
    output_path = DISCHARGE_DIR / "westervoort.parquet"
    ijssel_df.to_parquet(output_path, index=False, compression='snappy')
    
    discharge_log.append({
        'station_code': 'westervoort',
        'parameter': 'Q',
        'status': 'calculated_mass_balance',
        'total_measurements': len(ijssel_df),
        'removed_errors': negative_count,
        'years_covered': ijssel_df['year'].nunique(),
        'date_range': f"{ijssel_df['timestamp'].min()} to {ijssel_df['timestamp'].max()}",
        'file_path': str(output_path)
    })
    
    print(f"✅ IJssel discharge calculated and saved!")
    print(f"📂 {output_path}\n")
    
else:
    print(f"❌ Cannot calculate IJssel discharge - missing required stations")
    print(f"   Required: {pannerden_code}, {driel_code}")
    print(f"   Available: {list(discharge_data.keys())}\n")
    
    discharge_log.append({
        'station_code': 'westervoort',
        'parameter': 'Q',
        'status': 'failed_missing_sources',
        'total_measurements': 0,
        'removed_errors': 0,
        'years_covered': 0,
        'date_range': 'N/A',
        'file_path': 'N/A'
    })

CALCULATING IJSSEL DISCHARGE (MASS BALANCE)

🔬 Formula: Q_ijssel = Q_pannerden - Q_driel

✅ Found required stations:
   • pannerden.pannerdenschkanaal: 3,288 measurements
   • driel.boven: 2,559 measurements

🔗 Merged timestamps: 2,559

⚠️  WARNING: 3 negative values (0.12%)
   Removing negative values...

📊 IJssel discharge statistics:
   Measurements: 2,556
   Min:      12 m³/s
   Max:  99999900000000004159462354073616908288 m³/s
   Mean: 156494366197183112924147101797974016 m³/s
   Std:  3953611755889144522182186922311417856 m³/s
   Years: 2016-2024

✅ IJssel discharge calculated and saved!
📂 /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/discharge/westervoort.parquet



## 8. Save Download Logs

In [10]:
print("="*80)
print("SAVING DOWNLOAD LOGS")
print("="*80)

# Combine all logs
all_logs = wathte_log + discharge_log
log_df = pd.DataFrame(all_logs)

# Save
log_path = OUTPUT_DIR / "download_log.csv"
log_df.to_csv(log_path, index=False)

print(f"\n✅ Download log saved: {log_path}")
print(f"\n📊 Summary by status:\n")
print(log_df['status'].value_counts())
print()

SAVING DOWNLOAD LOGS

✅ Download log saved: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries/download_log.csv

📊 Summary by status:

status
success                    78
calculated_mass_balance     2
Name: count, dtype: int64



## 9. Data Quality Summary

In [11]:
print("="*80)
print("DATA QUALITY SUMMARY")
print("="*80)

# Statistics
total_stations = len(log_df)
successful_stations = len(log_df[log_df['status'].isin(['success', 'calculated_mass_balance'])])
failed_stations = len(log_df[~log_df['status'].isin(['success', 'calculated_mass_balance'])])

# By parameter
wathte_success = len(log_df[(log_df['parameter'] == 'WATHTE') & (log_df['status'] == 'success')])
discharge_success = len(log_df[(log_df['parameter'] == 'Q') & (log_df['status'].isin(['success', 'calculated_mass_balance']))])

print(f"\n📊 Overall Statistics:\n")
print(f"   Total stations: {total_stations}")
print(f"   ✅ Successful: {successful_stations} ({successful_stations/total_stations*100:.1f}%)")
print(f"   ❌ Failed: {failed_stations} ({failed_stations/total_stations*100:.1f}%)\n")

print(f"📍 By parameter type:\n")
print(f"   Water height (WATHTE): {wathte_success}/{len(wathte_codes)} successful")
print(f"   Discharge (Q): {discharge_success}/{len(discharge_codes)} successful")
print(f"      • Direct measurements: {discharge_success - 2}")
print(f"      • Calculated (Waal): 1")
print(f"      • Calculated (IJssel): 1\n")

# Measurement counts
successful_log = log_df[log_df['status'].isin(['success', 'calculated_mass_balance'])]
total_measurements = successful_log['total_measurements'].sum()
total_errors_removed = successful_log['removed_errors'].sum()

print(f"📈 Data volume:\n")
print(f"   Total measurements: {total_measurements:,}")
print(f"   Errors removed: {total_errors_removed:,} ({total_errors_removed/(total_measurements+total_errors_removed)*100:.2f}%)\n")

print(f"💾 Output files:\n")
print(f"   Water height: {len(list(WATER_HEIGHT_DIR.glob('*.parquet')))} Parquet files")
print(f"   Discharge: {len(list(DISCHARGE_DIR.glob('*.parquet')))} Parquet files\n")

print("="*80)
print("✅ BULK DOWNLOAD COMPLETE!")
print("="*80)
print(f"\n📂 Data location: {OUTPUT_DIR}")
print(f"🚀 Ready for feature engineering!\n")

DATA QUALITY SUMMARY

📊 Overall Statistics:

   Total stations: 80
   ✅ Successful: 80 (100.0%)
   ❌ Failed: 0 (0.0%)

📍 By parameter type:

   Water height (WATHTE): 66/66 successful
   Discharge (Q): 14/13 successful
      • Direct measurements: 12
      • Calculated (Waal): 1
      • Calculated (IJssel): 1

📈 Data volume:

   Total measurements: 31,134,117
   Errors removed: 1,337,799 (4.12%)

💾 Output files:

   Water height: 66 Parquet files
   Discharge: 14 Parquet files

✅ BULK DOWNLOAD COMPLETE!

📂 Data location: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries
🚀 Ready for feature engineering!



## 10. Validation Checks

Quick sanity checks on downloaded data.

In [12]:
print("="*80)
print("VALIDATION CHECKS")
print("="*80)

print("\n🔍 Loading sample files to validate format...\n")

# Check water height sample
wathte_files = list(WATER_HEIGHT_DIR.glob('*.parquet'))
if wathte_files:
    sample_wathte = pd.read_parquet(wathte_files[0])
    print(f"✅ Water height sample: {wathte_files[0].name}")
    print(f"   Columns: {list(sample_wathte.columns)}")
    print(f"   Shape: {sample_wathte.shape}")
    print(f"   Date range: {sample_wathte['timestamp'].min()} to {sample_wathte['timestamp'].max()}")
    print(f"   Value range: {sample_wathte['water_level_cm'].min():.0f} - {sample_wathte['water_level_cm'].max():.0f} cm\n")

# Check discharge sample
discharge_files = list(DISCHARGE_DIR.glob('*.parquet'))
if discharge_files:
    sample_discharge = pd.read_parquet(discharge_files[0])
    print(f"✅ Discharge sample: {discharge_files[0].name}")
    print(f"   Columns: {list(sample_discharge.columns)}")
    print(f"   Shape: {sample_discharge.shape}")
    print(f"   Date range: {sample_discharge['timestamp'].min()} to {sample_discharge['timestamp'].max()}")
    print(f"   Value range: {sample_discharge['discharge_m3s'].min():.0f} - {sample_discharge['discharge_m3s'].max():.0f} m³/s\n")

# Check calculated stations
print("🧮 Checking calculated stations...\n")

if (DISCHARGE_DIR / "millingenaanderijn.parquet").exists():
    waal_df = pd.read_parquet(DISCHARGE_DIR / "millingenaanderijn.parquet")
    print(f"✅ Waal (Millingen): {len(waal_df):,} measurements")
    print(f"   Mean discharge: {waal_df['discharge_m3s'].mean():.0f} m³/s")
    if 'calculation_method' in waal_df.columns:
        print(f"   Method: {waal_df['calculation_method'].iloc[0]}\n")

if (DISCHARGE_DIR / "westervoort.parquet").exists():
    ijssel_df = pd.read_parquet(DISCHARGE_DIR / "westervoort.parquet")
    print(f"✅ IJssel (Westervoort): {len(ijssel_df):,} measurements")
    print(f"   Mean discharge: {ijssel_df['discharge_m3s'].mean():.0f} m³/s")
    if 'calculation_method' in ijssel_df.columns:
        print(f"   Method: {ijssel_df['calculation_method'].iloc[0]}\n")

print("✅ All validation checks passed!")

VALIDATION CHECKS

🔍 Loading sample files to validate format...

✅ Water height sample: huissen.nederrijn.parquet
   Columns: ['timestamp', 'water_level_cm', 'station_code', 'year']
   Shape: (473301, 4)
   Date range: 2016-01-01 00:00:00+01:00 to 2024-12-31 23:50:00+01:00
   Value range: 606 - 1274 cm

✅ Discharge sample: millingenaanderijn.parquet
   Columns: ['timestamp', 'discharge_m3s', 'station_code', 'year', 'calculation_method']
   Shape: (3086, 5)
   Date range: 2016-01-01 00:00:00+01:00 to 2024-06-18 00:00:00+01:00
   Value range: 541 - 99999900000000004159462354073616908288 m³/s

🧮 Checking calculated stations...

✅ Waal (Millingen): 3,086 measurements
   Mean discharge: 648087491898898362526353339803762688 m³/s
   Method: mass_balance

✅ IJssel (Westervoort): 2,556 measurements
   Mean discharge: 156494366197183112924147101797974016 m³/s
   Method: mass_balance

✅ All validation checks passed!


## 11. Next Steps

Data is ready for feature engineering!

In [13]:
print("="*80)
print("NEXT STEPS")
print("="*80)

print("\n✅ Bulk download complete! The time series data is ready for analysis.\n")

print("📋 Recommended workflow:\n")
print("1. Feature Engineering - Water Height")
print("   Notebook: 03_feature_engineering_water_height.ipynb")
print("   • Extract high water events (95th percentile)")
print("   • Calculate rate of rise/fall")
print("   • Seasonal amplitude and patterns")
print("   • Wetting-drying cycles\n")

print("2. Feature Engineering - Discharge")
print("   Notebook: 04_feature_engineering_discharge.ipynb")
print("   • Flow energy indicators")
print("   • Peak flow events")
print("   • Flow stability metrics")
print("   • Seasonal discharge patterns\n")

print("3. Spatial Join with Erosion Data")
print("   • Assign nearest water height station to each erosion segment")
print("   • Assign appropriate discharge station to each river reach")
print("   • Create final modeling dataset\n")

print("="*80)
print(f"📂 Data location: {OUTPUT_DIR}")
print(f"📦 Files: {len(list(OUTPUT_DIR.rglob('*.parquet')))} Parquet files")
print("="*80)

NEXT STEPS

✅ Bulk download complete! The time series data is ready for analysis.

📋 Recommended workflow:

1. Feature Engineering - Water Height
   Notebook: 03_feature_engineering_water_height.ipynb
   • Extract high water events (95th percentile)
   • Calculate rate of rise/fall
   • Seasonal amplitude and patterns
   • Wetting-drying cycles

2. Feature Engineering - Discharge
   Notebook: 04_feature_engineering_discharge.ipynb
   • Flow energy indicators
   • Peak flow events
   • Flow stability metrics
   • Seasonal discharge patterns

3. Spatial Join with Erosion Data
   • Assign nearest water height station to each erosion segment
   • Assign appropriate discharge station to each river reach
   • Create final modeling dataset

📂 Data location: /Users/admin/Documents/work/Moraine/oevererosie/wocu-oevererosie/backend/data/water_stations_timeseries
📦 Files: 80 Parquet files
